In [ ]:
## Importing Libraries

import os
import torch
import random


import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print("Libraries Imported Successfully.")

In [ ]:
## Checking PyTorch and Graphics

print("PyTorch Version: ", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


In [ ]:
## Loading CSV Files

DATASET_PATH = r"Dataset"

TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")

print("Datasets Loaded Successfully.")

In [ ]:
## Converting Datasets into DataFrame

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Datasets converted to DataFrame.")

In [ ]:
## Pivoting Dataset

pivot_df = train_df.pivot(index="image_path", columns="target_name", values="target").reset_index()
display(pivot_df.head())

In [ ]:
## Creating Target Columns

TARGET_COLUMNS = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"]

In [ ]:
## Splitting Data in Train and Validation

train_data, valid_data = train_test_split(pivot_df, test_size = 0.20, random_state = 42, shuffle = True)

print("Number of Training Images: ", len(train_data))
print("Number of Validation Images: ", len(valid_data))

In [ ]:
## Setting Image Size

IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
BATCH_SIZE = 8
NUM_WORKERS = 0

In [ ]:
## Transforming Training Data

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
## Transforming Validation Data

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
## Creating BiomassDataset Class

class BiomassDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = os.path.join(DATASET_PATH, row["image_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row[TARGET_COLUMNS].values.astype(np.float32))

        return image, target

In [ ]:
## Creating Datasets

train_dataset = BiomassDataset(train_data, transform = train_transform)

valid_dataset = BiomassDataset(valid_data, transform = valid_transform)

In [ ]:
## Creating DataLoaders

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
## Verifying Datasets

print("Training Dataset :", len(train_dataset))
print("Validation Dataset :", len(valid_dataset))

In [ ]:
## Verifying Batch Shape

images, labels = next(iter(train_loader))
print("Image Batch Shape :", images.shape)
print("Label Shape :", labels.shape)

In [ ]:
## Visualizing Augmented Images

fig, axes = plt.subplots(2,4, figsize=(10, 5))

for i in range(8):
    img = images[i].permute(1,2,0).numpy()
    img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])

    img = np.clip(img,0,1)
    axes.flat[i].imshow(img)
    axes.flat[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## Dataset Summary

print("=" * 50)
print("Data Preprocessing Summary")
print("=" * 50)

print(f"Training Images      : {len(train_dataset)}")
print(f"Validation Images    : {len(valid_dataset)}")
print(f"Batch Size           : {BATCH_SIZE}")
print(f"Image Size           : {IMAGE_HEIGHT} x {IMAGE_WIDTH}")
print(f"Target Variables     : {len(TARGET_COLUMNS)}")